# 从零实现 BPE：训练、编码与工程边界

## 学习目标

完成本 notebook 后，你应该能够：

1. 区分 BPE 的训练、编码和解码三个阶段；
2. 正确统计带词频权重的相邻 pair，并处理重叠替换；
3. 解释为什么推理使用固定 merge rank，而不是重新统计输入；
4. 独立实现一个可运行的字符级 BPE，并分析 OOV、复杂度和工程取舍。

> 这里使用字符加词尾标记的教学版本。现代 LLM 常使用 byte-level BPE，但合并与 rank 的核心思想一致。


## 1. 原理与公式

设词 $w$ 当前表示为符号序列 $s_w=(s_1,\ldots,s_n)$，词频为 $f(w)$。相邻 pair $(a,b)$ 的加权频次为：

\[
C(a,b)=\sum_w f(w)\sum_{i=1}^{|s_w|-1}\mathbf{1}[s_i=a\land s_{i+1}=b].
\]

每轮选择 $\arg\max C(a,b)$，创建新符号 $ab$，并在所有序列中从左到右替换不重叠的出现。重复若干轮即可得到有序 merge 表。注意：频次相同的 pair 必须使用稳定 tie-break，否则同一语料可能得到不同词表。


In [ ]:
from collections import Counter

EOW = "</w>"
corpus = "low low low lower lower newest widest"
word_frequency = Counter(corpus.split())

# 用 tuple 作为不可变的当前分词；相同词只存一次，再乘词频。
initial_sequences = {
    tuple(list(word) + [EOW]): frequency
    for word, frequency in word_frequency.items()
}

def count_pairs(sequences):
    counts = Counter()
    for symbols, frequency in sequences.items():
        for left, right in zip(symbols, symbols[1:]):
            counts[(left, right)] += frequency
    return counts

print("词频:", word_frequency)
print("第一轮最高频 pair:", count_pairs(initial_sequences).most_common(5))


## 2. 不重叠合并

序列 `A A A` 中 `(A,A)` 虽有两个相邻位置，但它们共享中间元素。本轮从左到右合并只能得到 `AA A`，不能同时得到两个 `AA`。实现时匹配成功后指针前进 2，否则前进 1。

训练器下方使用 `(-频次, left, right)` 排序，既选最高频 pair，又让并列结果稳定。真实系统可选择其他 tie-break，但训练和复现必须一致。


In [ ]:
def merge_pair(symbols, pair, new_symbol):
    output = []
    i = 0
    while i < len(symbols):
        if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:
            output.append(new_symbol)
            i += 2
        else:
            output.append(symbols[i])
            i += 1
    return tuple(output)

def train_bpe(sequences, num_merges=10, min_frequency=1):
    sequences = dict(sequences)
    merges = []
    for rank in range(num_merges):
        counts = count_pairs(sequences)
        if not counts:
            break
        pair, frequency = min(counts.items(), key=lambda item: (-item[1], item[0]))
        if frequency < min_frequency:
            break
        new_symbol = "".join(pair)
        merges.append((pair, new_symbol, frequency))
        sequences = {
            merge_pair(symbols, pair, new_symbol): word_frequency
            for symbols, word_frequency in sequences.items()
        }
        print(f"rank={rank:02d}  {pair} -> {new_symbol!r}  count={frequency}")
    return merges, sequences

merges, trained_sequences = train_bpe(initial_sequences, num_merges=10)


## 3. 编码新文本：只使用训练好的 merge 顺序

编码新词时不能根据这个词重新选择最高频 pair。我们从字符和 `</w>` 开始，按训练 rank 依次应用规则。较晚规则依赖较早规则产生的符号，因此 merge 表的顺序是模型的一部分。

这个教学实现依次扫描全部规则，便于理解；生产实现通常用 pair rank、优先队列、链表和词级缓存减少重复扫描。


In [ ]:
def encode_word(word, merges):
    symbols = tuple(list(word) + [EOW])
    for pair, new_symbol, _ in merges:
        symbols = merge_pair(symbols, pair, new_symbol)
    return list(symbols)

def encode_text(text, merges):
    return [piece for word in text.split() for piece in encode_word(word, merges)]

def decode_pieces(pieces):
    return "".join(pieces).replace(EOW, " ").rstrip()

for sample in ["low", "lower", "lowest", "newest widest"]:
    pieces = encode_text(sample, merges)
    print(f"{sample!r:16} -> {pieces} -> {decode_pieces(pieces)!r}")


## 4. 边界测试与常见误区

- **重叠 pair**：替换必须不重叠；
- **忘记乘词频**：不同词型只统计一次会学出错误 merge；
- **推理重新计数**：会让同一词随上下文改变编码；
- **字符 OOV**：本实现无法表示训练字符集之外的字符；byte-level BPE 用 256 bytes 解决覆盖；
- **最长匹配等于 BPE**：不总成立，严格 BPE 依据 merge rank；
- **可逆性**：merge 可展开，但 `split()` 已丢失连续空格和换行，所以完整管线不严格可逆。


In [ ]:
# 最小边界测试
assert merge_pair(("A", "A", "A"), ("A", "A"), "AA") == ("AA", "A")
assert decode_pieces(encode_text("low lower", merges)) == "low lower"

known_characters = {char for word in word_frequency for char in word}
sample = "猫"
unknown = [char for char in sample if char not in known_characters]
print("字符级教学模型中的未知字符:", unknown)
print("UTF-8 byte 兜底则可表示为:", list(sample.encode("utf-8")))


## 5. 复杂度与相邻算法对比

朴素训练执行 $M$ 轮、每轮扫描约 $N$ 个当前符号，粗略为 $O(MN)$；生产训练器维护 pair 倒排位置与局部计数。朴素编码反复扫描可能接近 $O(L^2)$，可用优先队列和缓存优化。

| 方法 | 训练方向 | 推理核心 | 多切分 |
|---|---|---|---|
| BPE | 从小词表逐步加 merge | 按固定 merge rank 合并 | 标准版本无；可用 BPE-dropout |
| WordPiece | 按似然收益或实现近似扩词表 | 常见实现为最长匹配 | 标准版本无 |
| Unigram | 从大候选词表逐步剪枝 | lattice 上 Viterbi | 天然支持 n-best/采样 |


## 练习与面试总结

1. 将字符底座改为 UTF-8 bytes，并实现严格 byte round-trip。
2. 为训练器加入稳定 token ID、vocab 导出和断点续训。
3. 用堆和邻接链表优化编码，并与参考实现做随机对拍。
4. 分别训练 20、50、100 次 merge，比较序列长度和低频词碎片度。

**一分钟回答**：BPE 从字符或字节出发，每轮合并语料中最高频相邻 pair。训练保存有序 merges，推理只按 rank 重放，不能重新统计。它用高频长 token 和低频细粒度兜底，在词表参数与序列成本之间折中；byte-level 版本几乎无 OOV，但仍要处理预分词、可逆性、长尾膨胀和确定性工程问题。
